# Stock Machine Learning Trading Model (any ticker)
### Module 3 Project — Predicting a stock's N-day price direction to support daily positioning decisions

This notebook works for **any stock or ETF on Yahoo Finance**: US tickers (`AAPL`, `MSFT`, `NVDA`), ETFs (`SPY`), and other markets (`D05.SI` for DBS on SGX, `0700.HK`). Set `TICKER` in the configuration cell (§ Setup 1) and choose **Kernel → Restart & Run All**. All ticker-specific text, numbers, charts and the README are generated from the data.

| # | Assignment point | Section |
|---|---|---|
| 1 | Problem statement | §1 (a ticker-specific project card is printed in §3.1) |
| 2 | Type of ML problem | §2 |
| 3 | Dataset | §3 |
| 4 | EDA findings | §4 |
| 5 | Feature engineering & leakage control | §5 |
| 6 | Models tried (baseline → complex) | §6 |
| 7 | Hyperparameter tuning & findings | §7 |
| 8 | Evaluation metrics & error analysis | §8 |
| 9 | Opportunity sizing | §9 |
| 10 | Deploy / don't-deploy recommendation | §10 |
| 11 | Deployment plan | §11 |
| 12 | GitHub repo & README | Setup 0 writes `requirements.txt`; §12 generates `README.md` |
| + | Comparing several tickers | §13 (optional batch run) |

**Three ways to choose the stock**
1. Edit `TICKER = ...` in the configuration cell.
2. Set an environment variable before starting Jupyter: `TICKER=NVDA jupyter lab` (macOS/Linux) or `set TICKER=NVDA` (Windows).
3. Run several tickers automatically with §13.

## 1. Problem Statement

**Business context.** An investment desk (for example, a small fund or a robo-advisory team) holds a single stock as a core position. Each trading day it must decide how to position that stock for the coming holding period: **long, flat, or short**.

**Who is affected?** The portfolio manager who takes the decision, and the clients whose returns and drawdowns depend on it.

**Decision the model supports.** After the market closes on day *t*, the model estimates the probability that the stock will rise by more than the Buy threshold, fall by more than the Sell threshold, or stay in between over the next `FUTURE_PERIOD` trading days. A **confidence threshold** turns those probabilities into an action: Buy (long), Sell (short), or Hold (no position).

**How it is handled today (baseline).** The default is passive **buy-and-hold**: always long the stock, no timing. A machine-learning model is only worth deploying if, **after transaction costs**, it beats buy-and-hold on **risk-adjusted return** (Sharpe ratio) or materially reduces **drawdowns**.

**Success criteria (decided before modelling):**
1. Out-of-sample Sharpe ratio above buy-and-hold, after transaction costs.
2. Improvement that is unlikely to be luck (bootstrap confidence interval on excess return above 0).
3. Value created exceeds the cost of building and running the model.

*The exact ticker, horizon and thresholds for this run are printed in the **project card** in §3.1.*

## 2. Type of ML Problem

| Question | Answer | Why |
|---|---|---|
| Supervised or unsupervised? | **Supervised** | Each day has a known outcome (the realised N-day return), which becomes the label. |
| Classification or regression? | **Classification** | The desk acts on a direction (Buy / Hold / Sell), not an exact return. Predicting exact returns is much noisier. |
| Binary or multi-class? | **Multi-class (3 classes)** | Buy (+1), Hold (0), Sell (−1). Hold matters: it lets the model stay out when it is unsure. |
| Batch or real-time? | **Daily batch** | One prediction per day after the close; no intraday latency requirement. |
| Time series? | **Yes** | Rows are ordered in time, so we **never shuffle**: chronological splits and `TimeSeriesSplit` cross-validation only. |

### Setup 0: environment (run once)

The next cell writes `requirements.txt` for the GitHub repo; the cell after it installs the packages into **the kernel this notebook is running on**. If VS Code shows *"requires the ipykernel package"*, pick a kernel from an environment that has `ipykernel` (for example `dsai-m3`), or run `conda install -n <env> ipykernel`.

In [ ]:
%%writefile requirements.txt
numpy>=1.26
pandas>=2.1
matplotlib>=3.8
seaborn>=0.13
scikit-learn>=1.4
yfinance>=0.2.40
joblib>=1.3
jupyterlab>=4.0
ipykernel>=6.29
nbconvert>=7.0

: 

In [ ]:
import os, sys, subprocess
if not os.environ.get("BATCH_CHILD"):          # skipped during §13 batch runs (already installed)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)

### Setup 1: libraries and configuration

In [ ]:
import os
import re
import json
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from matplotlib.ticker import FuncFormatter, PercentFormatter, LogLocator

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, precision_score,
                             roc_auc_score, classification_report, confusion_matrix)

warnings.filterwarnings("ignore", category=FutureWarning)
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams.update({"figure.figsize": (12, 5), "font.size": 10, "axes.titleweight": "bold"})

COLORS = {"strategy": "#2563eb", "buyhold": "#6b7280", -1: "#dc2626", 0: "#9ca3af", 1: "#16a34a"}
CLASS_NAMES = {-1: "Sell (-1)", 0: "Hold (0)", 1: "Buy (1)"}

In [ ]:
# ==========================================
# ADJUSTABLE PROJECT PARAMETERS
# ==========================================
TICKER          = os.environ.get("TICKER", "TSLA").strip().upper()   # any Yahoo Finance symbol, e.g. "AAPL", "SPY", "D05.SI"
CURRENCY        = "$"            # symbol used in labels (e.g. "S$" for SGX stocks)

# Data period and chronological splits
START_DATE      = "2012-01-01"
END_DATE        = os.environ.get("END_DATE", "2026-10-01")   # exclusive
VAL_START       = "2021-01-01"   # train: START -> VAL_START | validation: VAL_START -> TEST_START | test: TEST_START -> END
TEST_START      = "2023-01-01"
SPLIT_MODE      = "auto"         # "dates" = always use the dates above
                                 # "auto"  = use the dates, but switch to a 60/20/20 split if the stock's history is too short
MIN_TRAIN_ROWS  = 750            # ~3 years of trading days needed for training before switching to 60/20/20
MIN_TEST_ROWS   = 120            # ~6 months needed for a meaningful test

# Target definition
FUTURE_PERIOD   = int(os.environ.get("HORIZON", 30))   # look-ahead horizon in trading days
THRESHOLD_MODE  = "auto"         # "auto"  = Buy/Sell band scaled to this stock's volatility (comparable across tickers)
                                 # "fixed" = use BUY_THRESHOLD / SELL_THRESHOLD below
AUTO_BAND_K     = 0.43           # auto band = K x std of N-day returns in the TRAINING period (~1/3 of days per class)
MIN_BAND        = 0.01           # never smaller than 1%, so a correct signal clears trading costs
BUY_THRESHOLD   = 0.06           # used when THRESHOLD_MODE = "fixed"
SELL_THRESHOLD  = -0.06

# Backtest assumptions
INITIAL_CAPITAL = 100_000.0
TRANSACTION_FEE = 0.001          # 0.1% per unit of position change (fees + slippage)
RISK_FREE_RATE  = 0.02           # annual, for Sharpe / Sortino
EXECUTION_LAG   = 2              # signal after close on day t, filled at close of t+1 (see §6)
LONG_ONLY       = False          # True = Sell signals mean "go flat" instead of "go short"
DEFAULT_CONFIDENCE = 0.45        # starting confidence threshold (tuned on validation in §7)
MIN_TRADES      = 6              # threshold search ignores settings with fewer position changes

# Modelling
RANDOM_STATE    = 42
N_CV_SPLITS     = 5
SELECTION_METRIC = "Sharpe"      # how the final model is chosen on validation: "Sharpe" or "Macro F1"
ALLOW_SYNTHETIC = True           # if the download fails: True = continue with synthetic data (code test only), False = stop

# Files: each ticker gets its own output folder, so results from several tickers can be compared (§13)
NOTEBOOK_FILE   = "stock_trading_model.ipynb"   # this notebook's file name (used by the §13 batch run)
IS_BATCH_CHILD  = bool(os.environ.get("BATCH_CHILD"))
SAFE_TICKER     = re.sub(r"[^A-Za-z0-9]+", "_", TICKER).strip("_")
OUTPUT_ROOT     = "outputs"
OUTPUT_DIR      = f"{OUTPUT_ROOT}/{SAFE_TICKER}"
FIG_DIR         = f"{OUTPUT_DIR}/figures"
DATA_DIR        = "data"
for d in (OUTPUT_DIR, FIG_DIR, DATA_DIR):
    os.makedirs(d, exist_ok=True)

def save_fig(name):
    plt.savefig(f"{FIG_DIR}/{name}.png", dpi=150, bbox_inches="tight")

def money(v):
    return f"{'-' if v < 0 else ''}{CURRENCY}{abs(v):,.0f}"

np.random.seed(RANDOM_STATE)
print(f"Ticker: {TICKER} | horizon {FUTURE_PERIOD} trading days | thresholds: {THRESHOLD_MODE} | "
      f"fee {TRANSACTION_FEE:.1%} | long-only={LONG_ONLY} | outputs -> {OUTPUT_DIR}/")

## 3. Dataset

**Source.** Daily OHLCV (open, high, low, close, volume) bars for the chosen ticker from Yahoo Finance via the `yfinance` library, **split- and dividend-adjusted** (`auto_adjust=True`). Adjustment matters: without it, a 4-for-1 stock split would look like a 75% one-day crash.

**Reproducibility.** The first successful download is cached to `data/<ticker>_<start>_<end>.csv`; later runs read the cache, so results do not change when Yahoo revises data. Delete the file to force a fresh download.

**Offline fallback.** If the download fails (no internet, or a **mistyped ticker**), the notebook can generate a **synthetic** random-walk series so the code still runs. Synthetic data has **no real patterns**, so its results must not be presented. A clear warning is printed whenever this happens; set `ALLOW_SYNTHETIC = False` to stop instead.

In [ ]:
CACHE_FILE = f"{DATA_DIR}/{SAFE_TICKER.lower()}_{START_DATE}_{END_DATE}.csv"

def download_yahoo():
    import yfinance as yf
    df = yf.download(TICKER, start=START_DATE, end=END_DATE, auto_adjust=True, progress=False)
    if df is None or df.empty:                    # yfinance often returns an EMPTY frame instead of raising
        raise ValueError(f"yfinance returned no rows for '{TICKER}' (check the symbol and your internet connection)")
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df[["Open", "High", "Low", "Close", "Volume"]].dropna()
    df = df[df["Close"] > 0]
    df.index = pd.to_datetime(df.index).tz_localize(None)
    df.index.name = "Date"
    return df

def make_synthetic():
    rng = np.random.default_rng(RANDOM_STATE + sum(map(ord, TICKER)))   # different random path per ticker
    dates = pd.bdate_range(START_DATE, END_DATE, inclusive="left")
    n = len(dates)
    rets = rng.normal(0.0008, 0.018, n)
    close = 15.0 * np.exp(np.cumsum(rets))
    high = close * (1 + np.abs(rng.normal(0, 0.008, n)))
    low = close * (1 - np.abs(rng.normal(0, 0.008, n)))
    open_ = low + (high - low) * rng.uniform(0.1, 0.9, n)
    vol = rng.integers(40_000_000, 120_000_000, n)
    return pd.DataFrame({"Open": open_, "High": high, "Low": low, "Close": close, "Volume": vol},
                        index=pd.DatetimeIndex(dates, name="Date"))

def load_data():
    if os.path.exists(CACHE_FILE):
        return pd.read_csv(CACHE_FILE, index_col="Date", parse_dates=True), "Yahoo Finance (cached)"
    try:
        df = download_yahoo()
        df.to_csv(CACHE_FILE)
        return df, "Yahoo Finance (downloaded)"
    except Exception as e:
        if not ALLOW_SYNTHETIC:
            raise RuntimeError(f"Download failed and ALLOW_SYNTHETIC=False. Reason: {e}")
        print(f"WARNING: could not download real data -> using SYNTHETIC data. Reason: {e}")
        return make_synthetic(), "SYNTHETIC (offline fallback)"

df_raw, DATA_SOURCE = load_data()
IS_SYNTHETIC = DATA_SOURCE.startswith("SYNTHETIC")
YEARS_OF_DATA = (df_raw.index.max() - df_raw.index.min()).days / 365.25

print(f"Ticker : {TICKER}")
print(f"Source : {DATA_SOURCE}")
print(f"Rows   : {len(df_raw):,} trading days   Columns: {list(df_raw.columns)}")
print(f"Period : {df_raw.index.min():%Y-%m-%d} to {df_raw.index.max():%Y-%m-%d} ({YEARS_OF_DATA:.1f} years)")
print(f"Missing values per column:\n{df_raw.isna().sum().to_string()}")
if df_raw.index.min() > pd.Timestamp(START_DATE) + pd.Timedelta(days=30):
    print(f"\nNote: {TICKER} data starts later than START_DATE (listed later?). Splits are checked in §3.1.")
if IS_SYNTHETIC:
    print("\n" + "!" * 80 + "\n  SYNTHETIC DATA: results below are for testing the code only. Re-run with internet access.\n" + "!" * 80)
df_raw.describe().round(2)

### 3.1 Splits, target variable and project card

For each day *t*, the future N-day return is

$$R_{t \to t+N} = \frac{\text{Close}_{t+N}}{\text{Close}_t} - 1, \qquad N = \texttt{FUTURE\_PERIOD}$$

| Label | Rule | Meaning |
|---|---|---|
| **Buy (+1)** | $R >$ Buy threshold | worth being long |
| **Sell (−1)** | $R <$ Sell threshold | worth being short / out |
| **Hold (0)** | otherwise | move too small to cover costs and risk |

**Why scale the thresholds to each stock (`THRESHOLD_MODE = "auto"`)?** A ±1.5% move over a week is big news for a utility stock but ordinary noise for a volatile growth stock. Using a fixed band would make one stock almost all Hold and another almost never Hold, so results could not be compared across tickers. In auto mode the band is

$$\text{band} = \max\left(K \times \sigma_N^{\text{train}},\; \texttt{MIN\_BAND}\right)$$

where $\sigma_N^{\text{train}}$ is the standard deviation of N-day returns **in the training period only** (using later periods would leak information). With $K = 0.43$, roughly a third of days fall into each class if returns are close to normal.

**Splits.** Train / validation / test follow the configured dates. If the stock has too little history before the validation date (for example a recent listing), `SPLIT_MODE = "auto"` switches to a **60% / 20% / 20% chronological split** of the available data. The last *N* rows have no future price yet, so they are dropped.

In [ ]:
# ---- 1. Decide the split dates for this ticker ----
WARMUP = 252                                         # rows needed for the longest feature (52-week high)
usable = df_raw.index[WARMUP:len(df_raw) - FUTURE_PERIOD]
if len(usable) < 500:
    raise ValueError(f"{TICKER}: only {len(usable)} usable days after warm-up. Choose a ticker with more history "
                     f"or an earlier START_DATE.")

n_train_by_date = (usable < pd.Timestamp(VAL_START)).sum()
n_test_by_date = (usable >= pd.Timestamp(TEST_START)).sum()
if SPLIT_MODE == "auto" and (n_train_by_date < MIN_TRAIN_ROWS or n_test_by_date < MIN_TEST_ROWS):
    VAL_START = str(usable[int(len(usable) * 0.6)].date())
    TEST_START = str(usable[int(len(usable) * 0.8)].date())
    SPLIT_USED = "60/20/20 (auto: history too short for the configured dates)"
else:
    SPLIT_USED = "configured dates"
print(f"Split used: {SPLIT_USED} | validation starts {VAL_START} | test starts {TEST_START}")

# ---- 2. Decide the Buy/Sell band from TRAINING data only ----
def future_return(close, horizon=FUTURE_PERIOD):
    return close.shift(-horizon) / close - 1.0                       # uses the FUTURE: for labels only

if THRESHOLD_MODE == "auto":
    train_close = df_raw["Close"][df_raw.index < pd.Timestamp(VAL_START)]
    sigma_n = future_return(train_close).dropna().std()
    band = max(AUTO_BAND_K * sigma_n, MIN_BAND)
    BUY_THRESHOLD, SELL_THRESHOLD = round(band, 4), -round(band, 4)
    print(f"Auto thresholds: {FUTURE_PERIOD}-day return std in training = {sigma_n:.2%} "
          f"-> band = {AUTO_BAND_K} x {sigma_n:.2%} = ±{band:.2%}")
else:
    print(f"Fixed thresholds: Buy > {BUY_THRESHOLD:+.2%}, Sell < {SELL_THRESHOLD:+.2%}")

# ---- 3. Build the target ----
def create_target(df, horizon, buy_thresh, sell_thresh):
    data = df.copy()
    data["Future_Return"] = future_return(data["Close"], horizon)
    data = data.dropna(subset=["Future_Return"])
    data["Target"] = np.select([data["Future_Return"] > buy_thresh,
                                data["Future_Return"] < sell_thresh], [1, -1], default=0)
    return data

df_target = create_target(df_raw, FUTURE_PERIOD, BUY_THRESHOLD, SELL_THRESHOLD)
dist = df_target["Target"].value_counts(normalize=True).sort_index().rename(CLASS_NAMES)
print("\nOverall class distribution:\n" + dist.map("{:.1%}".format).to_string())

# ---- 4. Project card (paste into slide 1) ----
print(f'''
================ PROJECT CARD ================
Stock            : {TICKER}  ({DATA_SOURCE})
Decision         : after each close, position {TICKER} for the next {FUTURE_PERIOD} trading days
Labels           : Buy if return > {BUY_THRESHOLD:+.2%}, Sell if < {SELL_THRESHOLD:+.2%}, else Hold
Baseline to beat : buy-and-hold {TICKER}, after {TRANSACTION_FEE:.1%} costs per trade
Primary metric   : out-of-sample Sharpe ratio; guardrail = max drawdown
==============================================''')

### 3.2 Known limitations

| Limitation | Impact | How we handle it |
|---|---|---|
| **One stock at a time** | Results for one ticker may not generalise to others. | §13 runs the same pipeline on several tickers and compares them. |
| **Price/volume data only** | No earnings, news, macro or sentiment data, so many drivers of returns are invisible. | Treat as a limitation; candidate future features. |
| **Low signal-to-noise** | Daily stock returns are close to random, so even good models have small edges. | Strong regularisation, simple baselines, honest out-of-sample testing. |
| **Class balance shifts over time** | Bull years have more Buys, bear years more Sells. | Class weights, macro-averaged metrics, error analysis by period. |
| **Overlapping labels** | Consecutive days share N−1 of their N future days, so neighbouring labels are correlated. Longer horizons make this stronger. | Purge gaps between splits and a `gap` in cross-validation (§5). |
| **Short histories** | Recently listed stocks have few years of data and may not cover a full market cycle. | Auto 60/20/20 split; interpret results with extra caution. |
| **Survivorship / hindsight bias** | Choosing a stock we already know did well inflates buy-and-hold. | Acknowledged; test a mix of winners and laggards in §13. |
| **Licensing** | Yahoo data is for personal/educational use. | Not redistributed; the README tells users how to download it themselves. |

## 4. Exploratory Data Analysis

Four key findings, each linked to a modelling decision. The wording describes what is typical for listed stocks; **check the printed numbers for your ticker** and adjust the wording on your slides if your stock behaves differently.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
axes[0].plot(df_raw.index, df_raw["Close"], color=COLORS["strategy"], lw=1.2)
axes[0].set_yscale("log")
axes[0].set_title(f"{TICKER} adjusted close (log scale)")
axes[0].set_ylabel(f"Price ({CURRENCY}, log)")
price_fmt = FuncFormatter(lambda v, _: f"{CURRENCY}{v:,.0f}" if v >= 10 else f"{CURRENCY}{v:,.2f}")
axes[0].yaxis.set_major_locator(LogLocator(base=10, subs=(1.0, 2.0, 3.0, 5.0, 7.0)))
axes[0].yaxis.set_major_formatter(price_fmt)
axes[0].yaxis.set_minor_formatter(FuncFormatter(lambda v, _: ""))
for d, lbl in [(VAL_START, "validation ->"), (TEST_START, "test ->")]:
    for ax in axes:
        ax.axvline(pd.Timestamp(d), color="black", ls=":", lw=1)
    axes[0].text(pd.Timestamp(d), axes[0].get_ylim()[1], " " + lbl, va="top", fontsize=9)

daily_ret = df_raw["Close"].pct_change()
roll_vol = daily_ret.rolling(20).std() * np.sqrt(252)
axes[1].plot(roll_vol.index, roll_vol, color=COLORS[-1], lw=1)
axes[1].set_title(f"{TICKER} 20-day rolling volatility (annualised)")
axes[1].set_ylabel("Volatility")
axes[1].yaxis.set_major_formatter(PercentFormatter(1.0))
plt.tight_layout(); save_fig("eda_1_price_and_volatility"); plt.show()

print(f"Price changed {df_raw['Close'].iloc[-1] / df_raw['Close'].iloc[0]:.1f}x over the period.")
print(f"Rolling volatility ranged from {roll_vol.min():.0%} to {roll_vol.max():.0%} (median {roll_vol.median():.0%}).")

**Finding 1: Prices are non-stationary; volatility comes in regimes.**
The price level drifts over the years, so a raw price (or anything in currency units) means something different at the start of the sample than at the end. Volatility is not constant: calm periods alternate with turbulent ones such as market sell-offs.

➡ **Modelling choice:** use only **scale-free features** (returns, ratios to moving averages, and MACD *divided by price*), and add **volatility features**. Error analysis in §8 is split by **volatility regime**.

In [ ]:
by_year = (df_target.groupby(df_target.index.year)["Target"]
           .value_counts(normalize=True).unstack().reindex(columns=[-1, 0, 1]).fillna(0))
fig, ax = plt.subplots(figsize=(13, 4.5))
bottom = np.zeros(len(by_year))
for c in [-1, 0, 1]:
    ax.bar(by_year.index.astype(str), by_year[c], bottom=bottom, color=COLORS[c], label=CLASS_NAMES[c], edgecolor="white")
    bottom += by_year[c].values
ax.set_title(f"{TICKER} class mix by year ({FUTURE_PERIOD}-day horizon, Buy > {BUY_THRESHOLD:+.1%}, Sell < {SELL_THRESHOLD:+.1%})")
ax.set_ylabel("Share of days"); ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.12))
plt.tight_layout(); save_fig("eda_2_class_mix_by_year"); plt.show()
print(by_year.rename(columns=CLASS_NAMES).map("{:.0%}".format).to_string())

**Finding 2: The class mix shifts from year to year.**
Bull years tilt towards Buy and bear years towards Sell. A model trained mostly on bullish years may over-predict Buy.

➡ **Modelling choice:** `class_weight="balanced"`; judge models on **macro F1** (every class counts equally) rather than accuracy; compare against a **majority-class baseline**; check performance **per period** in error analysis.

In [ ]:
lags = range(1, 11)
acf_ret = [daily_ret.autocorr(l) for l in lags]
acf_abs = [daily_ret.abs().autocorr(l) for l in lags]
conf = 1.96 / np.sqrt(daily_ret.notna().sum())

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
axes[0].bar(list(lags), acf_ret, color=COLORS["strategy"])
axes[0].set_title(f"{TICKER}: autocorrelation of daily returns")
axes[1].bar(list(lags), acf_abs, color=COLORS[-1])
axes[1].set_title(f"{TICKER}: autocorrelation of |daily returns| (volatility)")
for ax in axes:
    ax.axhspan(-conf, conf, color="grey", alpha=0.2, label="95% noise band")
    ax.set_xlabel("Lag (days)")
axes[0].legend()
plt.tight_layout(); save_fig("eda_3_autocorrelation"); plt.show()
print(f"Mean |autocorr| of returns, lags 1-10: {np.mean(np.abs(acf_ret)):.3f}  (noise band ±{conf:.3f})")
print(f"Mean autocorr of |returns|, lags 1-10: {np.mean(acf_abs):.3f}")

**Finding 3: Direction is close to unpredictable from past returns alone; volatility is usually more predictable.**
For most liquid stocks, return autocorrelations sit mostly inside the noise band, while absolute returns are positively autocorrelated (volatility clustering).

➡ **Modelling choice:** expect only a **small edge**. Use **regularised, shallow models** to avoid fitting noise; insist on beating simple **baselines**; use a **Hold** class and a **confidence threshold** so the model only trades when it is relatively sure.

## 5. Feature Engineering and Leakage Control

All features use **only information available at the close of day *t***: rolling windows and exponential averages look backwards, and nothing uses `shift(-k)`. Only the **target** looks forward. Every feature is **scale-free**, so the same feature set works for a $5 stock and a $500 stock.

| Group | Features | Reasoning |
|---|---|---|
| Momentum | `Return_1D/5D/10D/21D/63D` | Recent trend over a day, week, fortnight, month, quarter |
| Trend position | `Ratio_SMA10/50/200`, `Dist_52W_High` | Where price sits relative to its averages / yearly high |
| Oscillators | `RSI_14`, `BB_PctB` | Overbought / oversold signals |
| MACD (normalised) | `MACD_Pct`, `MACD_Signal_Pct`, `MACD_Hist_Pct` | Raw MACD is in currency units and grows with price (Finding 1), so it is divided by price |
| Volatility | `Vol_20D`, `Vol_Ratio_20_60`, `HL_Range_14` | Volatility level and whether it is rising; intraday range |
| Volume | `Volume_Ratio` | Unusual trading activity (relative to the stock's own average) |

**Cleaning.** Rows in the warm-up period (first ~252 days, needed for the 52-week high and 200-day average) are dropped; there are no other missing values. **Scaling** is applied only to Logistic Regression, inside a `Pipeline`, so the scaler is fitted on training folds only. Tree models do not need scaling. There are no categorical columns, so no encoding is needed.

In [ ]:
def build_features(df):
    d = df.copy()
    c = d["Close"]
    # Momentum
    for n in (1, 5, 10, 21, 63):
        d[f"Return_{n}D"] = c.pct_change(n)
    # Trend position
    for n in (10, 50, 200):
        d[f"Ratio_SMA{n}"] = c / c.rolling(n).mean()
    d["Dist_52W_High"] = c / c.rolling(252).max() - 1
    # RSI (14)
    delta = c.diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    d["RSI_14"] = 100 - 100 / (1 + gain / (loss + 1e-8))
    # MACD, normalised by price so it is comparable across years and tickers
    macd = c.ewm(span=12, adjust=False).mean() - c.ewm(span=26, adjust=False).mean()
    signal = macd.ewm(span=9, adjust=False).mean()
    d["MACD_Pct"], d["MACD_Signal_Pct"], d["MACD_Hist_Pct"] = macd / c, signal / c, (macd - signal) / c
    # Bollinger %B
    mid, sd = c.rolling(20).mean(), c.rolling(20).std()
    d["BB_PctB"] = (c - (mid - 2 * sd)) / (4 * sd + 1e-8)
    # Volatility
    r = c.pct_change()
    d["Vol_20D"] = r.rolling(20).std()
    d["Vol_Ratio_20_60"] = d["Vol_20D"] / (r.rolling(60).std() + 1e-8)
    d["HL_Range_14"] = ((d["High"] - d["Low"]) / c).rolling(14).mean()
    # Volume (some tickers, e.g. indices, report zero volume)
    vol_mean = d["Volume"].rolling(20).mean()
    d["Volume_Ratio"] = np.where(vol_mean > 0, d["Volume"] / (vol_mean + 1e-8), 1.0)
    return d.replace([np.inf, -np.inf], np.nan)

FEATURE_COLS = ["Return_1D", "Return_5D", "Return_10D", "Return_21D", "Return_63D",
                "Ratio_SMA10", "Ratio_SMA50", "Ratio_SMA200", "Dist_52W_High",
                "RSI_14", "MACD_Pct", "MACD_Signal_Pct", "MACD_Hist_Pct", "BB_PctB",
                "Vol_20D", "Vol_Ratio_20_60", "HL_Range_14", "Volume_Ratio"]

df_model = create_target(build_features(df_raw), FUTURE_PERIOD, BUY_THRESHOLD, SELL_THRESHOLD)
df_model = df_model.dropna(subset=FEATURE_COLS)
print(f"{TICKER} modelling table: {len(df_model):,} rows x {len(FEATURE_COLS)} features "
      f"({df_model.index.min():%Y-%m-%d} to {df_model.index.max():%Y-%m-%d})")

# Leakage self-check: features computed on data truncated at day t must equal features computed on the full history
t = df_model.index[len(df_model) // 2]
full_row = build_features(df_raw).loc[t, FEATURE_COLS]
trunc_row = build_features(df_raw.loc[:t]).loc[t, FEATURE_COLS]
assert np.allclose(full_row.values.astype(float), trunc_row.values.astype(float), equal_nan=True), "Look-ahead leakage detected!"
print(f"Leakage check passed: features on {t:%Y-%m-%d} are identical with or without future data.")

### 5.1 Chronological split with purging

Train → Validation → Test (the test set is sealed until §8). The exact dates for this ticker are shown in the table below.

**Why purge?** A label on the last training day depends on prices up to `FUTURE_PERIOD` days later, which fall inside the validation period. Keeping it in training would leak validation information. We therefore **drop the last `FUTURE_PERIOD` rows before each boundary**. Cross-validation uses `TimeSeriesSplit(gap=FUTURE_PERIOD)` for the same reason.

In [ ]:
def purged_index(df, end, start=None, purge=FUTURE_PERIOD):
    idx = df.index[df.index < pd.Timestamp(end)]
    if start is not None:
        idx = idx[idx >= pd.Timestamp(start)]
    return idx[:-purge] if purge > 0 else idx

train_idx = purged_index(df_model, VAL_START)
val_idx   = purged_index(df_model, TEST_START, start=VAL_START)
test_idx  = df_model.index[df_model.index >= pd.Timestamp(TEST_START)]
trval_idx = purged_index(df_model, TEST_START)          # used only for the final refit in §8

for name, idx in [("train", train_idx), ("validation", val_idx), ("test", test_idx)]:
    if len(idx) < 60:
        raise ValueError(f"The {name} set has only {len(idx)} rows. Adjust the dates, SPLIT_MODE or FUTURE_PERIOD.")

X_train, y_train = df_model.loc[train_idx, FEATURE_COLS], df_model.loc[train_idx, "Target"]
X_val,   y_val   = df_model.loc[val_idx,   FEATURE_COLS], df_model.loc[val_idx,   "Target"]
X_test,  y_test  = df_model.loc[test_idx,  FEATURE_COLS], df_model.loc[test_idx,  "Target"]

def period_label(idx):
    a, b = idx.min(), idx.max()
    return f"{a:%Y}" if a.year == b.year else f"{a:%Y}-{b:%y}" if a.year // 100 == b.year // 100 else f"{a:%Y}-{b:%Y}"

TRAIN_LABEL, VAL_LABEL, TEST_LABEL = period_label(train_idx), period_label(val_idx), period_label(test_idx)

split_summary = pd.DataFrame({
    "Rows": [len(train_idx), len(val_idx), len(test_idx)],
    "From": [i.min().date() for i in (train_idx, val_idx, test_idx)],
    "To":   [i.max().date() for i in (train_idx, val_idx, test_idx)],
    **{CLASS_NAMES[c]: [f"{(y == c).mean():.0%}" for y in (y_train, y_val, y_test)] for c in (-1, 0, 1)},
}, index=[f"Train ({TRAIN_LABEL})", f"Validation ({VAL_LABEL})", f"Test ({TEST_LABEL}, sealed)"])
print(f"{TICKER} split: {SPLIT_USED}")
split_summary

In [ ]:
# Feature diagnostics on TRAINING data only (looking at test data here would be a subtle form of leakage)
train_block = df_model.loc[train_idx, FEATURE_COLS + ["Future_Return"]]
fig, axes = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={"width_ratios": [1.3, 1]})
sns.heatmap(train_block[FEATURE_COLS].corr(), cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=axes[0],
            cbar_kws={"shrink": 0.7}, xticklabels=True, yticklabels=True)
axes[0].set_title(f"{TICKER}: feature correlation (train)")
corr_target = train_block.corr()["Future_Return"].drop("Future_Return").sort_values()
corr_target.plot(kind="barh", ax=axes[1], color=[COLORS[1] if v > 0 else COLORS[-1] for v in corr_target])
axes[1].set_title(f"Correlation with future {FUTURE_PERIOD}-day return (train)")
plt.tight_layout(); save_fig("eda_4_feature_correlations"); plt.show()

high_pairs = (train_block[FEATURE_COLS].corr().abs().where(lambda m: np.triu(np.ones(m.shape, bool), 1))
              .stack().loc[lambda s: s > 0.85].sort_values(ascending=False))
print(f"Strongest single-feature correlation with future return: {corr_target.abs().max():.3f}")
print(f"Highly correlated feature pairs (|r| > 0.85): {len(high_pairs)}")
print(high_pairs.round(2).to_string() if len(high_pairs) else "  none")

**Finding 4: Each feature alone usually has only a weak link with future returns, and several features overlap heavily** (for example the MACD variants and the short-term moving-average ratios).

➡ **Modelling choice:** no single rule will work, so we try models that **combine weak signals non-linearly** (Random Forest, Gradient Boosting). Overlapping features are harmless for trees; for Logistic Regression, **L2 regularisation** (tuned `C`) keeps coefficients stable.

## 6. Models Tried: from baselines to more complex models

**Evaluation helpers.**
- **Signals:** a non-Hold signal is issued only if its class probability is at least the confidence threshold *and* higher than the opposite class. Otherwise the model holds no position.
- **Backtest:** the signal is computed after the close on day *t* and **filled at the close of day *t*+1**, so the first return earned is from close *t*+1 to close *t*+2 (`EXECUTION_LAG = 2`). Each unit of position change pays `TRANSACTION_FEE`, so flipping from short to long pays it twice. Buy-and-hold pays one entry fee.
- **Metrics:** classification metrics (macro F1, balanced accuracy, per-class precision, ROC AUC) plus business metrics (Sharpe, CAGR, max drawdown, number of trades).

In [ ]:
def signals_from_proba(proba, classes, threshold):
    classes = list(classes)
    p_buy = proba[:, classes.index(1)] if 1 in classes else np.zeros(len(proba))
    p_sell = proba[:, classes.index(-1)] if -1 in classes else np.zeros(len(proba))
    sig = np.zeros(len(proba), dtype=int)
    sig[(p_buy >= threshold) & (p_buy > p_sell)] = 1
    sig[(p_sell >= threshold) & (p_sell > p_buy)] = -1
    return sig

def run_backtest(close, signals, lag=EXECUTION_LAG, fee=TRANSACTION_FEE, long_only=LONG_ONLY, capital=INITIAL_CAPITAL):
    bt = pd.DataFrame({"Close": close.values, "Signal": np.asarray(signals)}, index=close.index)
    if long_only:
        bt["Signal"] = bt["Signal"].clip(lower=0)
    bt["Position"] = bt["Signal"].shift(lag).fillna(0)
    bt["Stock_Return"] = bt["Close"].pct_change().fillna(0)
    bt["Turnover"] = bt["Position"].diff().abs().fillna(bt["Position"].abs())
    bt["Strategy_Return"] = bt["Position"] * bt["Stock_Return"] - bt["Turnover"] * fee
    bt["BuyHold_Return"] = bt["Stock_Return"]
    bt.iloc[0, bt.columns.get_loc("BuyHold_Return")] -= fee
    bt["Strategy_Equity"] = capital * (1 + bt["Strategy_Return"]).cumprod()
    bt["BuyHold_Equity"] = capital * (1 + bt["BuyHold_Return"]).cumprod()
    return bt

def financial_metrics(r, capital=INITIAL_CAPITAL, rf=RISK_FREE_RATE):
    r = pd.Series(r).fillna(0)
    equity = capital * (1 + r).cumprod()
    years = len(r) / 252
    sd = r.std()
    excess = r - rf / 252
    downside = np.sqrt((np.minimum(r, 0) ** 2).mean())
    return {
        "Total Return": equity.iloc[-1] / capital - 1,
        "CAGR": (equity.iloc[-1] / capital) ** (1 / years) - 1,
        "Volatility": sd * np.sqrt(252),
        "Sharpe": excess.mean() / sd * np.sqrt(252) if sd > 1e-12 else np.nan,
        "Sortino": excess.mean() / downside * np.sqrt(252) if downside > 1e-12 else np.nan,
        "Max Drawdown": (equity / equity.cummax() - 1).min(),
    }

def evaluate(model, X, y, idx, threshold, name=""):
    proba = model.predict_proba(X)
    sig = signals_from_proba(proba, model.classes_, threshold)
    bt = run_backtest(df_model.loc[idx, "Close"], sig)
    fm = financial_metrics(bt["Strategy_Return"])
    try:
        auc = roc_auc_score(y, proba, multi_class="ovr", average="macro", labels=list(model.classes_))
    except ValueError:
        auc = np.nan
    row = {
        "Model": name,
        "Accuracy": accuracy_score(y, sig),
        "Balanced Acc": balanced_accuracy_score(y, sig),
        "Macro F1": f1_score(y, sig, average="macro", labels=[-1, 0, 1], zero_division=0),
        "Buy Precision": precision_score(y, sig, labels=[1], average="macro", zero_division=0),
        "Sell Precision": precision_score(y, sig, labels=[-1], average="macro", zero_division=0),
        "ROC AUC (OvR)": auc,
        "Sharpe": fm["Sharpe"], "CAGR": fm["CAGR"], "Max Drawdown": fm["Max Drawdown"],
        "Trades": int((bt["Turnover"] > 0).sum()),
        "% Days in Market": (bt["Position"] != 0).mean(),
    }
    return row, sig, proba, bt

def buyhold_row(idx, name=f"Buy & Hold {TICKER} (current process)"):
    bt = run_backtest(df_model.loc[idx, "Close"], np.ones(len(idx), dtype=int))
    fm = financial_metrics(bt["BuyHold_Return"])
    return {"Model": name, "Sharpe": fm["Sharpe"], "CAGR": fm["CAGR"], "Max Drawdown": fm["Max Drawdown"],
            "Trades": 1, "% Days in Market": 1.0}

PCT_COLS = ["Accuracy", "Balanced Acc", "Macro F1", "Buy Precision", "Sell Precision", "CAGR",
            "Max Drawdown", "% Days in Market", "Train Macro F1"]
def show_table(rows):
    t = pd.DataFrame(rows).set_index("Model")
    fmt = {c: "{:.1%}" for c in PCT_COLS if c in t}
    fmt.update({c: "{:.2f}" for c in ("ROC AUC (OvR)", "Sharpe") if c in t})
    fmt.update({c: "{:.0f}" for c in ("Trades",) if c in t})
    return t, t.style.format(fmt, na_rep="–")

### 6.1 Candidate models (default settings), scored on the validation period

| Model | Role |
|---|---|
| Buy & Hold | Current process: the bar every model must beat |
| Majority-class `DummyClassifier` | "No-skill" ML baseline: always predicts the most common class |
| Logistic Regression (scaled, L2) | Simple, interpretable linear model |
| Random Forest (conservative settings) | Bagging ensemble with shallow trees and large leaves |
| Histogram Gradient Boosting | Boosting ensemble: usually strongest on tabular data |

`Train Macro F1` is shown next to the validation scores so that a large gap reveals **overfitting** (high variance).

In [ ]:
candidate_models = {
    "Majority class (Dummy)": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": Pipeline([("scaler", StandardScaler()),
                                     ("clf", LogisticRegression(C=1.0, class_weight="balanced", max_iter=5000))]),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=4, min_samples_leaf=25,
                                            class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200,
                                                        class_weight="balanced", random_state=RANDOM_STATE),
}

rows = [buyhold_row(val_idx)]
fitted_default = {}
for name, model in candidate_models.items():
    m = clone(model).fit(X_train, y_train)
    fitted_default[name] = m
    row, *_ = evaluate(m, X_val, y_val, val_idx, DEFAULT_CONFIDENCE, name)
    train_sig = signals_from_proba(m.predict_proba(X_train), m.classes_, DEFAULT_CONFIDENCE)
    row["Train Macro F1"] = f1_score(y_train, train_sig, average="macro", labels=[-1, 0, 1], zero_division=0)
    rows.append(row)

default_table, styled = show_table(rows)
default_table.to_csv(f"{OUTPUT_DIR}/model_comparison_default.csv")
print(f"{TICKER} | validation period {VAL_LABEL} | confidence threshold = {DEFAULT_CONFIDENCE}")
styled

**How to read this table:** a model only earns its complexity if it beats **both** baselines: higher macro F1 than the Dummy, and higher Sharpe (or smaller drawdown) than Buy & Hold. A big gap between `Train Macro F1` and `Macro F1` means the model memorised noise.

## 7. Hyperparameter Tuning

- **Method:** `GridSearchCV` with **`TimeSeriesSplit(n_splits=N_CV_SPLITS, gap=FUTURE_PERIOD)`** on the **training set only**: each fold trains on earlier years and validates on the next block, with a gap equal to the horizon so overlapping labels cannot leak.
- **Scoring:** macro F1 (treats Buy, Hold and Sell equally, matching Finding 2).
- **The test set stays sealed.** Model choice and confidence threshold are decided on the validation period.

In [ ]:
tscv = TimeSeriesSplit(n_splits=N_CV_SPLITS, gap=FUTURE_PERIOD)
print(f"Cross-validation: TimeSeriesSplit(n_splits={N_CV_SPLITS}, gap={FUTURE_PERIOD}) on {len(X_train):,} training rows\n")

search_spaces = {
    "Logistic Regression": (
        Pipeline([("scaler", StandardScaler()),
                  ("clf", LogisticRegression(class_weight="balanced", max_iter=5000))]),
        {"clf__C": [0.001, 0.01, 0.1, 1.0, 10.0]}),
    "Random Forest": (
        RandomForestClassifier(n_estimators=200, class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=1),
        {"max_depth": [3, 4, 6, 8], "min_samples_leaf": [10, 25, 50], "max_features": ["sqrt", 0.5]}),
    "Gradient Boosting": (
        HistGradientBoostingClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        {"learning_rate": [0.03, 0.1], "max_depth": [2, 3, 5], "max_iter": [100, 300], "l2_regularization": [0.0, 1.0]}),
}

searches = {}
for name, (est, grid) in search_spaces.items():
    gs = GridSearchCV(est, grid, cv=tscv, scoring="f1_macro", n_jobs=-1, return_train_score=True, error_score=np.nan)
    gs.fit(X_train, y_train)
    searches[name] = gs
    best = gs.cv_results_
    i = gs.best_index_
    print(f"{name:20s} best CV macro F1 = {gs.best_score_:.3f} (train {best['mean_train_score'][i]:.3f}, "
          f"fold std {best['std_test_score'][i]:.3f})  params = {gs.best_params_}")

In [ ]:
# Which hyperparameter mattered most? Effect = spread of mean CV score across that parameter's values
effects = []
for name, gs in searches.items():
    res = pd.DataFrame(gs.cv_results_)
    for p in gs.param_grid:
        by_val = res.groupby(res[f"param_{p}"].astype(str))["mean_test_score"].mean()
        effects.append({"Model": name, "Hyperparameter": p.replace("clf__", ""),
                        "Effect (max - min CV F1)": by_val.max() - by_val.min(),
                        "Best value": by_val.idxmax(), "Mean CV F1 by value": by_val.round(3).to_dict()})
effects = pd.DataFrame(effects).sort_values("Effect (max - min CV F1)", ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
labels = effects["Model"] + ": " + effects["Hyperparameter"]
ax.barh(labels[::-1], effects["Effect (max - min CV F1)"][::-1], color=COLORS["strategy"])
ax.set_title(f"{TICKER}: hyperparameter impact on cross-validated macro F1")
ax.set_xlabel("Change in mean CV macro F1 across tested values")
plt.tight_layout(); save_fig("tuning_hyperparameter_effects"); plt.show()
effects.reset_index(drop=True)

**Interpreting the tuning results.** In noisy financial data, the parameters that control **model complexity** (tree depth, minimum leaf size, regularisation strength `C`, learning rate) usually matter most. Shallower trees, larger leaves and stronger regularisation reduce **variance**: the model stops memorising day-to-day noise. Compare `best CV` with `train` scores above: a large gap is the overfitting signature from the bias–variance lesson.

In [ ]:
rows = [buyhold_row(val_idx), evaluate(fitted_default["Majority class (Dummy)"], X_val, y_val, val_idx,
                                       DEFAULT_CONFIDENCE, "Majority class (Dummy)")[0]]
for name, gs in searches.items():
    rows.append(evaluate(gs.best_estimator_, X_val, y_val, val_idx, DEFAULT_CONFIDENCE, f"{name} (tuned)")[0])
tuned_table, styled = show_table(rows)
tuned_table.to_csv(f"{OUTPUT_DIR}/model_comparison_tuned.csv")

ml_rows = tuned_table[tuned_table.index.str.contains("tuned")]
key = SELECTION_METRIC if ml_rows[SELECTION_METRIC].notna().any() else "Macro F1"
best_label = ml_rows[key].idxmax()
best_name = best_label.replace(" (tuned)", "")
best_model_trainfit = searches[best_name].best_estimator_
print(f"{TICKER}: selected on validation {key} -> {best_label}")
styled

### 7.1 Choosing the confidence threshold (on validation)

Like moving the 0.5 cut-off in the churn lesson, the confidence threshold trades **how often we trade** against **how reliable each trade is**. We sweep thresholds on the validation set and choose the one with the best Sharpe, requiring at least `MIN_TRADES` position changes so a lucky handful of trades cannot win.

In [ ]:
val_proba = best_model_trainfit.predict_proba(X_val)
sweep = []
for th in np.round(np.arange(0.34, 0.70, 0.02), 2):
    sig = signals_from_proba(val_proba, best_model_trainfit.classes_, th)
    bt = run_backtest(df_model.loc[val_idx, "Close"], sig)
    fm = financial_metrics(bt["Strategy_Return"])
    sweep.append({"Threshold": th, "Sharpe": fm["Sharpe"], "CAGR": fm["CAGR"],
                  "Macro F1": f1_score(y_val, sig, average="macro", labels=[-1, 0, 1], zero_division=0),
                  "Trades": int((bt["Turnover"] > 0).sum()), "% Days in Market": (bt["Position"] != 0).mean()})
sweep = pd.DataFrame(sweep)
eligible = sweep[(sweep["Trades"] >= MIN_TRADES) & sweep["Sharpe"].notna()]
BEST_THRESHOLD = float(eligible.loc[eligible["Sharpe"].idxmax(), "Threshold"]) if len(eligible) else DEFAULT_CONFIDENCE

fig, ax1 = plt.subplots(figsize=(12, 4.5))
ax1.plot(sweep["Threshold"], sweep["Sharpe"], "o-", color=COLORS["strategy"], label="Validation Sharpe")
ax1.axhline(buyhold_row(val_idx)["Sharpe"], color=COLORS["buyhold"], ls="--", label="Buy & Hold Sharpe")
ax1.axvline(BEST_THRESHOLD, color="black", ls=":", label=f"Chosen = {BEST_THRESHOLD:.2f}")
ax1.set_xlabel("Confidence threshold"); ax1.set_ylabel("Sharpe")
ax2 = ax1.twinx()
ax2.bar(sweep["Threshold"], sweep["% Days in Market"], width=0.015, alpha=0.25, color=COLORS[0], label="% days in market")
ax2.set_ylabel("% days in market"); ax2.grid(False); ax2.yaxis.set_major_formatter(PercentFormatter(1.0))
h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="best", fontsize=9)
ax1.set_title(f"{TICKER}: threshold sweep on validation ({VAL_LABEL}), {best_label}")
plt.tight_layout(); save_fig("tuning_threshold_sweep"); plt.show()
print(f"Chosen confidence threshold: {BEST_THRESHOLD:.2f}")
sweep.style.format({"Sharpe": "{:.2f}", "CAGR": "{:.1%}", "Macro F1": "{:.1%}", "% Days in Market": "{:.0%}"}, na_rep="–")

## 8. Final Evaluation on the Sealed Test Set

The chosen model and threshold are **refitted once on train + validation** (purged before the test start) and scored **once** on the test set. Nothing below is used to change the model.

**Metrics and why they fit the business problem**

| Metric | Why |
|---|---|
| **Sharpe ratio** (primary) | The desk cares about return per unit of risk, after costs, not raw accuracy. |
| **Max drawdown** | Clients feel losses from peaks; drawdown is a key risk limit. |
| **Buy / Sell precision** | When the model says "trade", how often is it right? False signals cost fees and money. |
| **Macro F1, balanced accuracy** | Classification quality with every class counted equally (accuracy alone can be fooled, as with the Dummy). |
| **ROC AUC (one-vs-rest)** | Threshold-free check of how well probabilities *rank* days. 0.5 means no skill. |

In [ ]:
final_model = clone(best_model_trainfit).fit(df_model.loc[trval_idx, FEATURE_COLS], df_model.loc[trval_idx, "Target"])
test_row, test_sig, test_proba, test_bt = evaluate(final_model, X_test, y_test, test_idx, BEST_THRESHOLD, best_label)
val_row = evaluate(best_model_trainfit, X_val, y_val, val_idx, BEST_THRESHOLD, best_label)[0]

strat_fm = financial_metrics(test_bt["Strategy_Return"])
bh_fm = financial_metrics(test_bt["BuyHold_Return"])

print(f"{TICKER} | model: {best_label} | threshold {BEST_THRESHOLD:.2f} | data: {DATA_SOURCE}\n")
print(classification_report(y_test, test_sig, labels=[-1, 0, 1],
                            target_names=[CLASS_NAMES[c] for c in (-1, 0, 1)], zero_division=0))

KEYS = ("Macro F1", "Buy Precision", "Sell Precision", "ROC AUC (OvR)", "Sharpe", "CAGR", "Max Drawdown", "Trades")
compare = pd.DataFrame({f"Validation ({VAL_LABEL})": {k: val_row[k] for k in KEYS},
                        f"Test ({TEST_LABEL})": {k: test_row[k] for k in KEYS}})
compare.to_csv(f"{OUTPUT_DIR}/validation_vs_test.csv")
compare

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [1, 1.6]})
cm = confusion_matrix(y_test, test_sig, labels=[-1, 0, 1])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[0],
            xticklabels=[CLASS_NAMES[c] for c in (-1, 0, 1)], yticklabels=[CLASS_NAMES[c] for c in (-1, 0, 1)])
axes[0].set_xlabel("Predicted (after threshold)"); axes[0].set_ylabel("Actual")
axes[0].set_title(f"{TICKER} confusion matrix: test ({TEST_LABEL})")

axes[1].plot(test_bt.index, test_bt["Strategy_Equity"], color=COLORS["strategy"], lw=2,
             label=f"ML strategy (Sharpe {strat_fm['Sharpe']:.2f})")
axes[1].plot(test_bt.index, test_bt["BuyHold_Equity"], color=COLORS["buyhold"], ls="--",
             label=f"Buy & Hold {TICKER} (Sharpe {bh_fm['Sharpe']:.2f})")
axes[1].set_title(f"{TICKER} test-period equity, {money(INITIAL_CAPITAL)} start, {TRANSACTION_FEE:.1%} fees")
axes[1].set_ylabel(f"Portfolio value ({CURRENCY})")
axes[1].yaxis.set_major_formatter(FuncFormatter(lambda v, _: money(v)))
axes[1].legend()
plt.tight_layout(); save_fig("test_confusion_and_equity"); plt.show()

fin_table = pd.DataFrame({"ML Strategy": strat_fm, f"Buy & Hold {TICKER}": bh_fm})
fin_table.loc["Trades"] = [test_row["Trades"], 1]
fin_table.loc["% Days in Market"] = [test_row["% Days in Market"], 1.0]
fin_table.to_csv(f"{OUTPUT_DIR}/test_financial_metrics.csv")
styler = fin_table.style
for r in fin_table.index:
    styler = styler.format({"Sharpe": "{:.2f}", "Sortino": "{:.2f}", "Trades": "{:.0f}"}.get(r, "{:.1%}"),
                           subset=pd.IndexSlice[[r], :])
styler

### 8.1 Error analysis: where does the model fail?

1. **By volatility regime** (cut-offs set on *training* data): does the model only work in calm markets?
2. **By quarter**: is performance stable, or driven by one lucky period?
3. **Signal quality**: what actually happened after Buy and Sell signals, and how often did the model call the exact opposite direction?

In [ ]:
ea = df_model.loc[test_idx, ["Close", "Future_Return", "Target", "Vol_20D"]].copy()
ea["Signal"] = test_sig
ea["Strategy_Return"] = test_bt["Strategy_Return"]
ea["BuyHold_Return"] = test_bt["BuyHold_Return"]

lo, hi = df_model.loc[train_idx, "Vol_20D"].quantile([1 / 3, 2 / 3])
ea["Vol regime"] = pd.cut(ea["Vol_20D"], [-np.inf, lo, hi, np.inf], labels=["Low vol", "Mid vol", "High vol"])

def segment_stats(g):
    return pd.Series({
        "Days": len(g),
        "Macro F1": f1_score(g["Target"], g["Signal"], average="macro", labels=[-1, 0, 1], zero_division=0),
        "Strategy ann. return": g["Strategy_Return"].mean() * 252,
        "Buy&Hold ann. return": g["BuyHold_Return"].mean() * 252,
    })

by_regime = ea.groupby("Vol regime", observed=False).apply(segment_stats)
by_quarter = ea.groupby(ea.index.to_period("Q")).apply(segment_stats)
by_regime.to_csv(f"{OUTPUT_DIR}/error_by_regime.csv"); by_quarter.to_csv(f"{OUTPUT_DIR}/error_by_quarter.csv")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
by_regime[["Strategy ann. return", "Buy&Hold ann. return"]].plot(kind="bar", ax=axes[0], color=[COLORS["strategy"], COLORS["buyhold"]], rot=0)
axes[0].set_title(f"{TICKER}: annualised return by volatility regime (test)"); axes[0].axhline(0, color="black", lw=0.8)
axes[0].yaxis.set_major_formatter(PercentFormatter(1.0))
by_quarter["Macro F1"].plot(kind="bar", ax=axes[1], color=COLORS["strategy"], rot=45)
axes[1].axhline(1 / 3, color=COLORS[-1], ls="--", label="≈ random guessing (1/3)")
axes[1].set_title(f"{TICKER}: macro F1 by quarter (test)"); axes[1].legend()
plt.tight_layout(); save_fig("error_analysis_segments"); plt.show()

signal_quality = pd.DataFrame({
    CLASS_NAMES[s]: {
        "Signals issued": int((ea["Signal"] == s).sum()),
        f"Avg future {FUTURE_PERIOD}d return": ea.loc[ea["Signal"] == s, "Future_Return"].mean(),
        "% future return > 0": (ea.loc[ea["Signal"] == s, "Future_Return"] > 0).mean(),
        "Opposite-direction errors": int(((ea["Signal"] == s) & (ea["Target"] == -s)).sum()) if s != 0 else np.nan,
    } for s in (1, 0, -1)}).T
print(f"{TICKER} signal quality (test):")
display(signal_quality)
display(by_regime.style.format({"Macro F1": "{:.1%}", "Strategy ann. return": "{:.1%}", "Buy&Hold ann. return": "{:.1%}", "Days": "{:.0f}"}))

In [ ]:
# What drives the predictions? Permutation importance on VALIDATION (model fitted on train only)
pi = permutation_importance(best_model_trainfit, X_val, y_val, scoring="f1_macro", n_repeats=10,
                            random_state=RANDOM_STATE, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=FEATURE_COLS).sort_values()
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(imp.index, imp.values, xerr=pd.Series(pi.importances_std, index=FEATURE_COLS)[imp.index],
        color=[COLORS["strategy"] if v > 0 else COLORS[0] for v in imp.values])
ax.axvline(0, color="black", lw=0.8)
ax.set_title(f"{TICKER}: permutation importance on validation, {best_label}")
ax.set_xlabel("Drop in macro F1 when the feature is shuffled")
plt.tight_layout(); save_fig("feature_importance"); plt.show()
TOP_FEATURES = list(imp.sort_values(ascending=False).index[:5])
print("Top 5 features:", ", ".join(TOP_FEATURES))

**How to talk about errors in the presentation:** name the regime or quarter where the model loses most, show the opposite-direction error count (the costliest mistakes: calling Buy before a fall), and note that importance shows **association, not causation**.

## 9. Opportunity Sizing

The value of the model is the **extra return over the current process (buy-and-hold)**, scaled to the capital the desk would actually allocate, minus the cost of building and running it. Every assumption is a parameter below; change them to match your story.

In [ ]:
# ---- Assumptions (edit these) ----
AUM               = 1_000_000   # capital the desk would run with the model
BUILD_HOURS, HOURLY_RATE = 120, 80          # one-off build effort
ANNUAL_DATA_COST  = 1_200       # market-data licence per year
ANNUAL_INFRA_COST = 600         # cloud compute + storage per year
ANNUAL_MAINT_HOURS = 40         # monitoring, retraining, reviews (hours/yr)
N_BOOT, BLOCK = 2000, 20        # block bootstrap settings for the confidence interval

build_cost = BUILD_HOURS * HOURLY_RATE
annual_run_cost = ANNUAL_DATA_COST + ANNUAL_INFRA_COST + ANNUAL_MAINT_HOURS * HOURLY_RATE

excess_cagr = strat_fm["CAGR"] - bh_fm["CAGR"]
annual_value = excess_cagr * AUM
year1_net = annual_value - build_cost - annual_run_cost
breakeven_aum = (annual_run_cost + build_cost) / excess_cagr if excess_cagr > 0 else np.nan
drawdown_saving = (strat_fm["Max Drawdown"] - bh_fm["Max Drawdown"]) * AUM   # positive = smaller worst loss

# Block bootstrap of daily excess returns (keeps volatility clusters together)
rng = np.random.default_rng(RANDOM_STATE)
diff = (test_bt["Strategy_Return"] - test_bt["BuyHold_Return"]).values
n = len(diff)
boot = np.empty(N_BOOT)
for b in range(N_BOOT):
    starts = rng.integers(0, max(n - BLOCK, 1), size=int(np.ceil(n / BLOCK)))
    sample = np.concatenate([diff[s:s + BLOCK] for s in starts])[:n]
    boot[b] = sample.mean() * 252
ci_low, ci_high = np.percentile(boot, [2.5, 97.5])
p_positive = (boot > 0).mean()

sizing = pd.DataFrame.from_dict({
    f"Test-period excess CAGR vs Buy & Hold {TICKER}": f"{excess_cagr:+.2%}",
    "95% CI of annual excess return (bootstrap)": f"[{ci_low:+.2%}, {ci_high:+.2%}]",
    "Probability excess return > 0": f"{p_positive:.0%}",
    f"Annual value on {money(AUM)} AUM": money(annual_value),
    "Change in worst drawdown on AUM": money(drawdown_saving),
    "One-off build cost": money(build_cost),
    "Annual running cost": money(annual_run_cost),
    "Year-1 net value": money(year1_net),
    "Break-even AUM (year 1)": money(breakeven_aum) if np.isfinite(breakeven_aum) else "n/a (no excess return)",
}, orient="index", columns=["Value"])
sizing.to_csv(f"{OUTPUT_DIR}/opportunity_sizing.csv")
sizing

**Reading the numbers:** a positive point estimate is not enough. If the 95% interval includes zero, the test period cannot distinguish skill from luck, which is common with a single stock over a short window.

## 10. Recommendation: Deploy or Not Deploy

In [ ]:
dummy_val_f1 = evaluate(fitted_default["Majority class (Dummy)"], X_val, y_val, val_idx, BEST_THRESHOLD)[0]["Macro F1"]
checks = {
    "Uses real market data (not synthetic)": not IS_SYNTHETIC,
    f"Test Sharpe beats Buy & Hold {TICKER}": bool(np.nan_to_num(strat_fm["Sharpe"], nan=-9) > bh_fm["Sharpe"]),
    "Beats the no-skill Dummy on macro F1 (validation)": bool(val_row["Macro F1"] > dummy_val_f1),
    "Excess return statistically above zero (CI lower bound > 0)": bool(ci_low > 0),
    "Year-1 value exceeds build + running cost": bool(year1_net > 0),
}
print(f"{TICKER} go / no-go checks:")
for k, v in checks.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")

if IS_SYNTHETIC:
    RECOMMENDATION = "NO DECISION: results use synthetic data; re-run with real market data"
elif all(checks.values()):
    RECOMMENDATION = "DEPLOY to paper trading (shadow mode), then a small-capital A/B test"
elif checks[f"Test Sharpe beats Buy & Hold {TICKER}"]:
    RECOMMENDATION = "DO NOT DEPLOY YET: promising, but the edge is not yet statistically reliable or cost-justified; run paper trading"
else:
    RECOMMENDATION = f"DO NOT DEPLOY: the model does not beat buy-and-hold {TICKER} after costs"
print(f"\nRECOMMENDATION ({TICKER}) -> {RECOMMENDATION}")

**Risks and limitations to state alongside the recommendation**

| Risk | Why it matters | Mitigation |
|---|---|---|
| **Overfitting to history** | Many settings were tried; some apparent skill may be luck. | Sealed test set, purged CV, confidence interval, paper trading before real money. |
| **Regime change / drift** | Market behaviour changes (rates, sector cycles, regulation). | Drift monitoring and scheduled retraining (§11). |
| **Short selling risk** | Shorts have unlimited loss and borrowing costs not modelled here. | Set `LONG_ONLY = True` for a more conservative version. |
| **Costs underestimated** | Slippage can exceed the assumed fee in volatile or illiquid stocks. | Re-run the backtest with 2–3× the fee as a stress test. |
| **Single-stock concentration** | Results depend on one company. | Compare several tickers (§13) before scaling capital. |
| **Regulatory / suitability** | Automated signals given to clients may count as investment advice (e.g. MAS rules in Singapore). | Human sign-off on trades; compliance review before client use. |

*"Do not deploy yet" is a valid and often the most credible answer for a trading model, as long as the evidence above supports it.*

## 11. Deployment Plan

### 11.1 Architecture

```
 ┌──────────────┐   daily after close  ┌──────────────────────┐    ┌───────────────────────┐
 │ Market data  │ ───────────────────▶ │ Ingestion job        │──▶ │ Feature pipeline      │
 │ (Yahoo /     │                      │ (scheduler: Dagster/ │    │ build_features()      │
 │  paid vendor)│                      │  cron) + data checks │    │ + leakage assertions  │
 └──────────────┘                      └──────────────────────┘    └──────────┬────────────┘
                                                                              │ features (day t)
                                      ┌──────────────────────┐    ┌───────────▼────────────┐
                                      │ Model registry       │──▶ │ Batch prediction       │
                                      │ one model per ticker │    │ predict_proba + thresh │
                                      │ (.joblib + threshold)│    │ -> Buy / Hold / Sell   │
                                      └──────────▲───────────┘    └───────────┬────────────┘
                                                 │ retrain monthly            │ signal + probability
 ┌──────────────────────┐             ┌──────────┴───────────┐    ┌───────────▼────────────┐
 │ Monitoring dashboard │ ◀────────── │ Monitoring job       │ ◀──│ Signal store / API     │
 │ drift, hit rate,     │   alerts    │ PSI drift, rolling   │    │ (FastAPI / Streamlit)  │
 │ Sharpe, drawdown     │             │ Sharpe, drawdown     │    │ -> PM review -> broker │
 └──────────────────────┘             └──────────────────────┘    └────────────────────────┘
```

Each ticker has its own model, threshold and Buy/Sell band, stored together in `outputs/<TICKER>/final_model.joblib`, so new tickers can be added without changing the code.

### 11.2 Retraining and drift detection
- **Retrain monthly** on an expanding window, using the same purged `TimeSeriesSplit` procedure; **re-tune quarterly**. A new version is promoted only if it beats the current version on the most recent hold-out window.
- **Data drift:** Population Stability Index (PSI) of each feature versus training data. PSI > 0.25 = significant drift → investigate / retrain.
- **Performance drift:** rolling 60-day Buy/Sell precision and rolling Sharpe versus buy-and-hold. Alert if precision falls to random-guess level for 4 consecutive weeks.
- **Hard risk limit:** if strategy drawdown exceeds 15%, automatically revert to buy-and-hold (kill switch).

### 11.3 A/B test before full rollout
| Item | Plan |
|---|---|
| Phase 1 | **Shadow mode / paper trading** for 3 months: signals logged, no money at risk. |
| Control (A) | Current process: buy-and-hold on half of the pilot capital. |
| Treatment (B) | ML signals on the other half (same start date, same fees). |
| Primary metric | Net Sharpe ratio over 6 months; guardrail: max drawdown. |
| Roll out if | B's Sharpe > A's, B's drawdown ≤ A's, and bootstrap CI of excess return > 0. |
| Roll back if | B's drawdown > 15%, or B trails A by > 5% return for two consecutive months. |

In [ ]:
# Monitoring demo: PSI of each feature in the test period vs. training data
def psi(expected, actual, bins=10):
    cuts = np.unique(np.quantile(expected, np.linspace(0, 1, bins + 1)))
    cuts[0], cuts[-1] = -np.inf, np.inf
    e = np.histogram(expected, cuts)[0] / len(expected)
    a = np.histogram(actual, cuts)[0] / len(actual)
    e, a = np.clip(e, 1e-6, None), np.clip(a, 1e-6, None)
    return float(np.sum((a - e) * np.log(a / e)))

drift = pd.Series({f: psi(X_train[f].values, X_test[f].values) for f in FEATURE_COLS}).sort_values(ascending=False)
drift_df = pd.DataFrame({"PSI": drift, "Status": pd.cut(drift, [-np.inf, 0.1, 0.25, np.inf],
                                                        labels=["Stable", "Watch", "Drift - investigate"])})
drift_df.to_csv(f"{OUTPUT_DIR}/feature_drift_psi.csv")
print(f"{TICKER}: feature drift, test vs train")
drift_df.style.format({"PSI": "{:.3f}"})

## 12. Save Artifacts for the GitHub Repository

Saves the trained model, test signals, a machine-readable `results.json` (used by §13) and a `results_summary.md` for this ticker.

In [ ]:
joblib.dump({"ticker": TICKER, "model": final_model, "threshold": BEST_THRESHOLD, "features": FEATURE_COLS,
             "horizon": FUTURE_PERIOD, "buy_threshold": BUY_THRESHOLD, "sell_threshold": SELL_THRESHOLD},
            f"{OUTPUT_DIR}/final_model.joblib")
ea[["Close", "Signal", "Target", "Future_Return", "Strategy_Return"]].to_csv(f"{OUTPUT_DIR}/test_signals.csv")

results = {
    "Ticker": TICKER, "Data": DATA_SOURCE, "Synthetic": IS_SYNTHETIC,
    "Horizon (days)": FUTURE_PERIOD, "Band": f"±{BUY_THRESHOLD:.1%}" if BUY_THRESHOLD == -SELL_THRESHOLD else f"{BUY_THRESHOLD:+.1%}/{SELL_THRESHOLD:+.1%}",
    "Split": SPLIT_USED.split(" (")[0], "Test period": TEST_LABEL, "Model": best_label,
    "Confidence": BEST_THRESHOLD,
    "Strategy CAGR": strat_fm["CAGR"], "B&H CAGR": bh_fm["CAGR"],
    "Strategy Sharpe": strat_fm["Sharpe"], "B&H Sharpe": bh_fm["Sharpe"],
    "Strategy MaxDD": strat_fm["Max Drawdown"], "B&H MaxDD": bh_fm["Max Drawdown"],
    "Macro F1": test_row["Macro F1"], "Trades": test_row["Trades"],
    "CI low": ci_low, "CI high": ci_high, "Recommendation": RECOMMENDATION,
}
with open(f"{OUTPUT_DIR}/results.json", "w") as f:
    json.dump({k: (None if isinstance(v, float) and not np.isfinite(v) else v) for k, v in results.items()}, f, indent=2, default=str)

summary = f'''# Results summary ({TICKER})

- **Data:** {DATA_SOURCE}, {df_model.index.min():%Y-%m-%d} to {df_model.index.max():%Y-%m-%d}, {len(df_model):,} modelling rows, {len(FEATURE_COLS)} features
- **Target:** {FUTURE_PERIOD}-day return > {BUY_THRESHOLD:+.1%} = Buy, < {SELL_THRESHOLD:+.1%} = Sell, else Hold ({THRESHOLD_MODE} thresholds)
- **Splits:** train {TRAIN_LABEL} · validation {VAL_LABEL} · test {TEST_LABEL} ({SPLIT_USED})
- **Selected model:** {best_label}, params {searches[best_name].best_params_}, confidence threshold {BEST_THRESHOLD:.2f}

| Test {TEST_LABEL} | ML strategy | Buy & Hold {TICKER} |
|---|---|---|
| CAGR | {strat_fm['CAGR']:.1%} | {bh_fm['CAGR']:.1%} |
| Sharpe | {strat_fm['Sharpe']:.2f} | {bh_fm['Sharpe']:.2f} |
| Max drawdown | {strat_fm['Max Drawdown']:.1%} | {bh_fm['Max Drawdown']:.1%} |
| Macro F1 | {test_row['Macro F1']:.1%} | – |
| Trades | {test_row['Trades']} | 1 |

- **Top features:** {", ".join(TOP_FEATURES)}
- **Excess return 95% CI:** [{ci_low:+.2%}, {ci_high:+.2%}] per year
- **Recommendation:** {RECOMMENDATION}
'''
with open(f"{OUTPUT_DIR}/results_summary.md", "w") as f:
    f.write(summary)
print(summary)
print(f"Saved to {OUTPUT_DIR}/:", sorted(os.listdir(OUTPUT_DIR)))

## 13. Comparing Several Tickers (optional)

The whole notebook is a reusable pipeline, so it can be re-run for a list of tickers. Each run saves its results to `outputs/<TICKER>/`, and the table below collects every ticker that has been run so far.

**Two ways to fill the table**
- **Manual:** change `TICKER` in the configuration cell and choose *Restart & Run All*, once per ticker.
- **Automatic:** set `RUN_BATCH = True` below. The notebook re-executes itself once per ticker in `BATCH_TICKERS` (about 2–4 minutes each). Make sure `NOTEBOOK_FILE` in the configuration cell matches this file's name.

**Fair comparison:** keep the same horizon, `THRESHOLD_MODE`, dates and fees for every ticker. Auto thresholds make the Buy/Sell bands comparable across stocks with different volatility.

In [ ]:
BATCH_TICKERS = ["AAPL", "MSFT", "NVDA", "SPY", "D05.SI"]   # edit: any Yahoo Finance symbols
RUN_BATCH     = False                                      # True = run every ticker above now

if RUN_BATCH and not IS_BATCH_CHILD:
    if not os.path.exists(NOTEBOOK_FILE):
        raise FileNotFoundError(f"Set NOTEBOOK_FILE to this notebook's file name (currently '{NOTEBOOK_FILE}').")
    for t in BATCH_TICKERS:
        t_safe = re.sub(r"[^A-Za-z0-9]+", "_", t.upper()).strip("_")
        if t.upper() == TICKER:
            print(f"{t}: already run in this session, skipped")
            continue
        print(f"{t}: running ...", flush=True)
        os.makedirs(f"{OUTPUT_ROOT}/{t_safe}", exist_ok=True)
        proc = subprocess.run(
            [sys.executable, "-m", "jupyter", "nbconvert", "--to", "notebook", "--execute", NOTEBOOK_FILE,
             "--output", f"run_{t_safe}.ipynb", "--output-dir", f"{OUTPUT_ROOT}/{t_safe}",
             "--ExecutePreprocessor.timeout=3600"],
            env={**os.environ, "TICKER": t, "BATCH_CHILD": "1"}, capture_output=True, text=True)
        print("   done" if proc.returncode == 0 else f"   FAILED:\n{proc.stderr[-800:]}")
elif not IS_BATCH_CHILD:
    print("RUN_BATCH = False: showing tickers already run (set RUN_BATCH = True to run the list).")

In [ ]:
all_results = []
for path in sorted(glob.glob(f"{OUTPUT_ROOT}/*/results.json")):
    with open(path) as f:
        all_results.append(json.load(f))
comparison = pd.DataFrame(all_results).set_index("Ticker") if all_results else pd.DataFrame()

if len(comparison):
    comparison.to_csv(f"{OUTPUT_ROOT}/ticker_comparison.csv")
    if comparison["Synthetic"].any():
        print("WARNING: some rows use SYNTHETIC data:", ", ".join(comparison.index[comparison["Synthetic"]]))
    if comparison["Horizon (days)"].nunique() > 1:
        print("WARNING: tickers were run with different horizons; comparisons are not like-for-like.")

    if len(comparison) > 1 and not IS_BATCH_CHILD:
        fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
        comparison[["Strategy Sharpe", "B&H Sharpe"]].plot(kind="bar", ax=axes[0], rot=0,
                                                           color=[COLORS["strategy"], COLORS["buyhold"]])
        axes[0].set_title("Test Sharpe: ML strategy vs buy-and-hold"); axes[0].axhline(0, color="black", lw=0.8)
        ci = comparison[["CI low", "CI high"]].astype(float)
        mid = (ci["CI low"] + ci["CI high"]) / 2
        axes[1].errorbar(range(len(ci)), mid, yerr=[mid - ci["CI low"], ci["CI high"] - mid], fmt="o",
                         color=COLORS["strategy"], capsize=5)
        axes[1].axhline(0, color=COLORS[-1], ls="--", label="no edge")
        axes[1].set_xticks(range(len(ci))); axes[1].set_xticklabels(ci.index)
        axes[1].yaxis.set_major_formatter(PercentFormatter(1.0))
        axes[1].set_title("95% CI of annual excess return vs buy-and-hold"); axes[1].legend()
        plt.tight_layout(); plt.savefig(f"{OUTPUT_ROOT}/ticker_comparison.png", dpi=150, bbox_inches="tight"); plt.show()

    show_cols = ["Data", "Horizon (days)", "Band", "Test period", "Model", "Strategy CAGR", "B&H CAGR",
                 "Strategy Sharpe", "B&H Sharpe", "Strategy MaxDD", "B&H MaxDD", "Macro F1", "Recommendation"]
    display(comparison[show_cols].style.format(
        {c: "{:.1%}" for c in ("Strategy CAGR", "B&H CAGR", "Strategy MaxDD", "B&H MaxDD", "Macro F1")} |
        {c: "{:.2f}" for c in ("Strategy Sharpe", "B&H Sharpe")}, na_rep="–"))
else:
    print("No results found yet.")

**What to look for:** does the model beat buy-and-hold on **most** tickers or only one? An edge that appears on one stock but not on similar ones is more likely to be luck. Stocks where buy-and-hold had a huge run are especially hard to beat.

## 14. Generate `README.md` and publish

This cell writes the repository README using the results above (including the multi-ticker table if more than one ticker has been run). Replace `<your-username>/<your-repo>` with your GitHub details before pushing.

In [ ]:
if IS_BATCH_CHILD:
    print("Batch child run: README not written.")
else:
    multi = ""
    if len(comparison) > 1:
        t = comparison[["Horizon (days)", "Band", "Test period", "Strategy Sharpe", "B&H Sharpe",
                        "Strategy CAGR", "B&H CAGR", "Recommendation"]].copy()
        for c in ("Strategy Sharpe", "B&H Sharpe"):
            t[c] = t[c].map(lambda v: "–" if v is None or pd.isna(v) else f"{v:.2f}")
        for c in ("Strategy CAGR", "B&H CAGR"):
            t[c] = t[c].map(lambda v: "–" if v is None or pd.isna(v) else f"{v:.1%}")
        t["Recommendation"] = t["Recommendation"].str.split(":").str[0]
        header = "| Ticker | " + " | ".join(t.columns) + " |\n|" + "---|" * (len(t.columns) + 1) + "\n"
        rows_md = "".join(f"| {i} | " + " | ".join(str(v) for v in r) + " |\n" for i, r in t.iterrows())
        multi = "\n### All tickers run so far\n\n" + header + rows_md + "\n![Ticker comparison](outputs/ticker_comparison.png)\n"

    readme = f'''# Stock ML Trading Model: Predicting N-Day Price Direction for Any Ticker

Module 3 Project (SCTP Advanced Professional Certificate in Data Science & AI)

## Project overview and problem statement

An investment desk holds a single stock and must decide each day whether to be **long, flat or short** for the coming holding period. Today it simply **buys and holds**. This project builds a reusable machine-learning pipeline that, for any Yahoo Finance ticker, predicts whether the stock's next N-day return will be:

| Label | Rule |
|---|---|
| Buy (+1) | N-day return above the Buy threshold |
| Hold (0) | between the thresholds |
| Sell (−1) | N-day return below the Sell threshold |

Thresholds can be fixed or scaled automatically to each stock's volatility, so results are comparable across tickers. The model's probabilities, filtered by a confidence threshold, become daily trading signals, judged against buy-and-hold on **after-cost Sharpe ratio** and drawdown, followed by a deploy / don't-deploy recommendation.

**ML problem type:** supervised, multi-class classification on time-series data, with daily batch predictions.

## Dataset

- **Source:** Yahoo Finance daily OHLCV, split- and dividend-adjusted, via the [`yfinance`](https://pypi.org/project/yfinance/) library
- **Default period:** {START_DATE} to {END_DATE}; recently listed stocks automatically use a 60/20/20 chronological split
- **Features:** 18 scale-free technical features (returns, moving-average ratios, RSI, price-normalised MACD, Bollinger %B, volatility and volume ratios)
- **How to obtain it:** the notebook downloads the data on first run and caches it in `data/`. Yahoo data is for personal and educational use, so the CSVs are not committed to this repository.

> If a download fails (no internet or a mistyped ticker), the notebook falls back to **synthetic** data and prints a warning. Synthetic results have no real-world meaning.

## Setup and reproducing the results

```bash
git clone https://github.com/<your-username>/<your-repo>.git
cd <your-repo>
conda create -n stock-ml python=3.11 -y
conda activate stock-ml
pip install -r requirements.txt
python -m ipykernel install --user --name stock-ml
jupyter lab {NOTEBOOK_FILE}        # set TICKER in the configuration cell, then Kernel -> Restart & Run All
```

Run another ticker without editing the notebook (macOS/Linux):

```bash
TICKER=NVDA jupyter nbconvert --to notebook --execute {NOTEBOOK_FILE} --output run_NVDA.ipynb
```

Or set `RUN_BATCH = True` in §13 to run a list of tickers and build a comparison table. Runtime is about 2–4 minutes per ticker.

## Method summary

1. **Baselines:** buy-and-hold (the current process) and a majority-class `DummyClassifier`
2. **Models:** Logistic Regression (scaled, L2), Random Forest, Histogram Gradient Boosting, all with balanced class weights
3. **Tuning:** `GridSearchCV` with `TimeSeriesSplit(n_splits={N_CV_SPLITS}, gap=horizon)` on training data, scored by macro F1
4. **Model and confidence-threshold selection** on the validation period
5. **Single final test** after refitting on train + validation
6. **Backtest:** signal after the close on day *t*, filled at the close of *t*+1, {TRANSACTION_FEE:.1%} cost per unit of position change
7. **Error analysis** by volatility regime and quarter; permutation feature importance
8. **Opportunity sizing** with a block-bootstrap confidence interval on excess return

**Leakage controls:** backward-looking features only (verified by an automated check), Buy/Sell band estimated on training data only, purge gaps between splits, time-series cross-validation with a gap, and a scaler fitted inside a `Pipeline`.

## Results

### Latest run: {TICKER}

{summary.split(chr(10), 1)[1].strip()}
{multi}
## Repository structure

```
.
├── {NOTEBOOK_FILE}        # full analysis (sections follow the assignment brief)
├── requirements.txt
├── README.md
├── data/                             # cached Yahoo downloads (created on first run, git-ignored)
└── outputs/
    ├── ticker_comparison.csv/.png    # all tickers run so far (§13)
    └── <TICKER>/                     # one folder per ticker
        ├── figures/                  # PNG charts for slides
        ├── model_comparison_*.csv, validation_vs_test.csv, test_financial_metrics.csv
        ├── error_by_regime.csv, error_by_quarter.csv, opportunity_sizing.csv, feature_drift_psi.csv
        ├── test_signals.csv, final_model.joblib
        └── results.json, results_summary.md
```

## Deployment architecture (proposed)

```mermaid
flowchart LR
    A[Market data<br/>daily after close] --> B[Ingestion job<br/>+ data checks]
    B --> C[Feature pipeline]
    C --> D[Batch prediction<br/>model + threshold per ticker]
    R[(Model registry)] --> D
    D --> E[Signal store / API]
    E --> F[PM review → broker<br/>paper or live]
    E --> G[Monitoring<br/>PSI drift, rolling Sharpe,<br/>drawdown kill switch]
    G -->|monthly retrain| R
```

## Disclaimer

Educational project only. This is not investment advice.
'''
    with open("README.md", "w") as f:
        f.write(readme)
    print(f"README.md written ({len(readme):,} characters). Files ready for GitHub:")
    for p in [NOTEBOOK_FILE, "README.md", "requirements.txt", "outputs/"]:
        print("  ", p, "✓" if os.path.exists(p) else "(missing: check NOTEBOOK_FILE)")
    if IS_SYNTHETIC or (len(comparison) and comparison["Synthetic"].any()):
        print("\nWARNING: some results come from SYNTHETIC data. Re-run with internet access before publishing.")

### Publish to GitHub

```bash
echo -e "data/\n.ipynb_checkpoints/\n__pycache__/\noutputs/*/run_*.ipynb" > .gitignore   # Yahoo data is not redistributed
git init && git add . && git commit -m "Module 3 project: stock ML trading model"
git branch -M main
git remote add origin https://github.com/<your-username>/<your-repo>.git
git push -u origin main
```

## Lessons Learned

1. **Accuracy is a lie here too.** A majority-class model can look accurate; macro F1, precision on trade signals, and above all **after-cost Sharpe vs buy-and-hold** are the honest measures.
2. **Time-series discipline matters more than model choice:** no shuffling, purge gaps, `TimeSeriesSplit` with `gap`, execution one day after the signal, and a test set opened only once.
3. **Simple, regularised models are competitive on noisy data.** Tuning showed complexity controls (depth, leaf size, regularisation) matter most: a bias–variance story.
4. **The confidence threshold is a business lever.** A higher threshold means fewer, more selective trades and lower fee drag, at the cost of time out of the market.
5. **One stock is not evidence.** An edge should hold across several comparable tickers (§13), and statistical uncertainty (bootstrap CI) must be part of the go/no-go decision.